In [6]:

import pandas as pd
import time
import requests
import json

import pymssql

In [5]:
defect = pd.read_csv("defect.csv")

defect.head()

,name,defectid
0,LJ6406953A42C000000000000A044FC012078410,6b35f021556c4126a88592949ed40479_10.200.218.127
1,LJ6406953A42C000000000000A044FC012078810,9c5436add6ac4534ab3985de43e46e11_10.200.218.127
2,LJ6406953A42C000000000000A044FC012079510,6159b4a921a24a68a64c97fb1c0e2365_10.200.218.127
3,LJ6406953A42C000000000000A044FC012078010,8fe1fdb4ab1c485683c4bc38358fcdca_10.200.218.127
4,LJ6406953A42C000000000000A044FC012140210,8785c42e0a5247a48fed9daf6a268965_10.200.218.127


In [7]:
def get_conn(host, user, password, database):
    """
    Creates a connection to a database using the provided host, username, password, and database name.

    Parameters:
        host (str): The host name or IP address of the database server.
        user (str): The username used to authenticate with the database server.
        password (str): The password used to authenticate with the database server.
        database (str): The name of the database to connect to.

    Returns:
        conn (pymssql.Connection): A connection object representing the connection to the database.
    """
    try:
        conn = pymssql.connect(
            server=host,
            user=user,
            password=password,
            database=database,
            as_dict=True,
        )
    # 捕获数据库连接错误
    except Exception as e:
        print(e)

    return conn
Host = "10.80.65.109"
User = "datalab"
Password = "cx3x#x&W"
Database = "IAData"
conn =  get_conn(Host,User,Password,Database)

In [12]:
sql = f""" SELECT defectid,FVBucketName,FVFileName FROM dbo.defect where defectid in {tuple(defect["defectid"].values)}	"""

cursor = conn.cursor()
cursor.execute(sql)
channelkeys = cursor.fetchall()

channelkeys

[{'defectid': '0189c2e8c30847ef80f13bf9ad9d329e_10.200.218.127',
  'FVBucketName': 'sa-42c-202406',
  'FVFileName': '4e916f288f0642dbb51081627191a646'},
 {'defectid': '054f3e32c68e4cd7b3f0581aea22ffde_10.200.218.127',
  'FVBucketName': 'sa-42c-202406',
  'FVFileName': 'e793206a34f54e32a17d63e241d9ad5b'},
 {'defectid': '22c58011b9ad4fa68068a97a74eb5ba4_10.200.218.127',
  'FVBucketName': 'sa-42c-202406',
  'FVFileName': '0e804167ac434796bdefd0bf8d14001c'},
 {'defectid': '233ad76c96a440c2abe2250e50fe78c3_10.200.218.127',
  'FVBucketName': 'sa-42c-202406',
  'FVFileName': 'd75830fde0b04668888bc38e8940096d'},
 {'defectid': '2af7a7a27d444b7fb51cec78ec89ef09_10.200.218.127',
  'FVBucketName': 'sa-42c-202406',
  'FVFileName': 'b3e7bdbe0e4442febe7670264f938dfe'},
 {'defectid': '2bfbe09239ed46419478663dccb3d5c8_10.200.218.127',
  'FVBucketName': 'sa-42c-202406',
  'FVFileName': '3c0b7aabbba24e9b972e4ca59b0e4c2d'},
 {'defectid': '2bfe85eb5ff647ec8f1fda31ad75edb2_10.200.218.127',
  'FVBucketName':

In [13]:
def get_json(defects):
    """
    Retrieves JSON data from a list of defects.

    Args:
        defects (list): A list of dictionaries representing defects.

    Returns:
        pandas.DataFrame: A DataFrame containing the JSON data from the defects.
    """
    # logger.info(f"defects点列表的长度：{len(defects)}")

    json_data_list = []
    start = time.time()

    for d in defects:
        try:
            FVBucketName = d["FVBucketName"]
            FVFileName = d["FVFileName"]

            url = f"http://10.211.89.15:9008/SAS3API/api/Storage/sa/{FVBucketName}/{FVFileName}?filetype=json"
            response = requests.get(url, timeout=10)
            json_data = json.loads(response.text)
            # json_data["frameid"] = d["frameid"]
            json_data["defectid"] = d["defectid"]
            # json_data["label"] = d["Defect_RE_result"]
            json_data_list.append(json_data)
        except requests.exceptions.Timeout:
            print("请求超时")
        except json.decoder.JSONDecodeError:
            print("JSON解码错误")
            print(f"FVBucketName, FVFileName: {FVBucketName}, {FVFileName}")
            print(response.text)
        except Exception as e:
            print(e)

    df = pd.DataFrame(json_data_list)

    print(f"程序运行时间：{time.time() - start} s")

    return df
df =   get_json(channelkeys)

程序运行时间：0.4289276599884033 s


In [14]:
df.hea

,flawArea,flawCustomType,flawLength,flawWidth,area,roundness,numConnected,circularity,rect2Phi,rect2Len1,...,regionbackgroundMeangraydiff,regionmeanwidth,regionlinearity,defectid,highgraydiffarea2defectareaRatio1,highgraydiffarea2defectareaRatio2,highgraydiffarea2defectareaRatio3,highgraydiffarea2defectareaRatio4,highgraydiffarea2defectareaRatio5,
0,0.11,S1V1-线性划伤,1.05,0.29,744,0.45356,2,0.104809,57.6526,87.3687,...,65.0387,8.51564,0.888864,0189c2e8c30847ef80f13bf9ad9d329e_10.200.218.127,NaN,NaN,NaN,NaN,NaN,NaN
1,0.01,S1V2-残胶,0.13,0.13,84,0.830317,1,0.789229,45,10.8995,...,68.4338,7.70678,0.012168,054f3e32c68e4cd7b3f0581aea22ffde_10.200.218.127,0.952268,0.845137,0.738007,0.583264,0.49994,NaN
2,0.02,S2V5-VA油墨点,0.22,0.15,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,22c58011b9ad4fa68068a97a74eb5ba4_10.200.218.127,NaN,NaN,NaN,NaN,NaN,
3,0.01,S2V5-VA油墨点,0.15,0.10,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,233ad76c96a440c2abe2250e50fe78c3_10.200.218.127,NaN,NaN,NaN,NaN,NaN,
4,0.04,S1V2-视窗刺伤,0.35,0.20,304,0.68487,1,0.416205,33.6901,29.0124,...,87.4418,10.4783,0.307392,2af7a7a27d444b7fb51cec78ec89ef09_10.200.218.127,0.996678,0.98681,0.947337,0.868392,0.759843,NaN


In [15]:
final_data = pd.merge(defect, df, on="defectid")
final_data.head()

,name,defectid,flawArea,flawCustomType,flawLength,flawWidth,area,roundness,numConnected,circularity,...,area2smallestRectRatio,regionbackgroundMeangraydiff,regionmeanwidth,regionlinearity,highgraydiffarea2defectareaRatio1,highgraydiffarea2defectareaRatio2,highgraydiffarea2defectareaRatio3,highgraydiffarea2defectareaRatio4,highgraydiffarea2defectareaRatio5,
0,LJ6406953A42C000000000000A044FC012078410,6b35f021556c4126a88592949ed40479_10.200.218.127,0.01,S1V2-残胶,0.12,0.10,56,0.778979,1,NaN,...,0.7,59.4149,5.6,0.0234132,0.85699,0.714158,0.553473,0.499911,0.482057,NaN
1,LJ6406953A42C000000000000A044FC012078810,9c5436add6ac4534ab3985de43e46e11_10.200.218.127,0.01,S1V2-残胶,0.12,0.10,60,0.69191,1,NaN,...,0.75,31.8291,6,0.0575301,0.783203,0.566572,0.349942,0.29995,0.183303,NaN
2,LJ6406953A42C000000000000A044FC012079510,6159b4a921a24a68a64c97fb1c0e2365_10.200.218.127,0.01,S1V2-残胶,0.10,0.07,44,0.772208,1,0.714302,...,0.916667,63.5769,5.5,0.0979077,0.74983,0.545331,0.43172,0.363554,0.340832,NaN
3,LJ6406953A42C000000000000A044FC012078010,8fe1fdb4ab1c485683c4bc38358fcdca_10.200.218.127,0.01,S1V2-残胶,0.14,0.10,80,0.79758,1,NaN,...,0.833333,64.8855,6.66667,0.148714,0.96238,0.837395,0.699913,0.612423,0.474941,NaN
4,LJ6406953A42C000000000000A044FC012140210,8785c42e0a5247a48fed9daf6a268965_10.200.218.127,0.02,S1V1-线性划伤,0.35,0.13,140,0.568718,2,0.18565,...,0.453953,56.3301,4.75646,0.754549,0.885651,0.742804,0.542818,0.449968,0.328548,NaN


In [17]:
final_data.to_csv("final_data.csv",index=False,encoding="utf-8-sig")